<a href="https://colab.research.google.com/github/Jishnuc25/hiver-project/blob/main/hiver_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# ================================================================
# HIVER SDE INTERN ASSIGNMENT
# COMPLETE LOCAL AI SUPPORT AGENT
#
# NO OPENAI API
# NO API KEY
# NO EXTERNAL LLM
#
# Pipeline:
# Dataset
#    ↓
# Data Analysis
#    ↓
# Brand Selection
#    ↓
# Intent Definition
#    ↓
# Intent Classification
#    ↓
# Historical Retrieval
#    ↓
# Reply Generation
#    ↓
# Auto Handle / Escalate
#    ↓
# Golden Set
#    ↓
# Evaluation
#    ↓
# Failure Analysis
#    ↓
# README + ZIP
# ================================================================


# ================================================================
# 0. INSTALL LIBRARIES
# ================================================================

!pip -q install pandas numpy scikit-learn tqdm joblib


# ================================================================
# 1. IMPORT LIBRARIES
# ================================================================

import os
import re
import json
import shutil
import warnings
import numpy as np
import pandas as pd

from collections import Counter

from tqdm.auto import tqdm

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

import joblib

warnings.filterwarnings("ignore")


print("=" * 70)
print("HIVER SDE INTERN ASSIGNMENT")
print("LOCAL AI CUSTOMER SUPPORT AGENT")
print("NO OPENAI API")
print("=" * 70)


# ================================================================
# STEP 1 — GET DATASET
# ================================================================

print("\n")
print("=" * 70)
print("STEP 1 — GET DATASET")
print("=" * 70)

from google.colab import files

print("Upload your CSV dataset.")

uploaded = files.upload()

csv_files = [
    x for x in uploaded.keys()
    if x.lower().endswith(".csv")
]

if len(csv_files) == 0:
    raise ValueError("Please upload a CSV file.")

CSV_FILE = csv_files[0]

print("\nSelected dataset:")
print(CSV_FILE)


# ================================================================
# LOAD CSV
# ================================================================

def load_csv(path):

    encodings = [
        "utf-8",
        "utf-8-sig",
        "latin1",
        "cp1252"
    ]

    for encoding in encodings:

        try:

            data = pd.read_csv(
                path,
                encoding=encoding,
                low_memory=False
            )

            print(
                "Loaded using:",
                encoding
            )

            return data

        except Exception:
            continue

    raise ValueError(
        "Unable to read CSV."
    )


df = load_csv(
    CSV_FILE
)

print(
    "\nDataset shape:",
    df.shape
)


# ================================================================
# STEP 2 — ANALYZE DATA
# ================================================================

print("\n")
print("=" * 70)
print("STEP 2 — ANALYZE DATA")
print("=" * 70)

print(
    "\nRows:",
    len(df)
)

print(
    "Columns:",
    len(df.columns)
)

print("\nColumn names:")

for col in df.columns:

    print(
        "-",
        col
    )


print("\nMissing values:")

print(
    df.isnull()
      .sum()
      .sort_values(
          ascending=False
      )
)


print(
    "\nDuplicate rows:",
    df.duplicated().sum()
)


# ================================================================
# AUTOMATIC COLUMN DETECTION
# ================================================================

def detect_column(
    columns,
    keywords
):

    # Exact match first

    for keyword in keywords:

        for col in columns:

            if (
                str(col)
                .lower()
                .strip()
                ==
                keyword.lower()
            ):

                return col


    # Partial match

    for keyword in keywords:

        for col in columns:

            if (
                keyword.lower()
                in
                str(col).lower()
            ):

                return col


    return None


MESSAGE_KEYWORDS = [
    "message",
    "text",
    "tweet",
    "review_text",
    "review",
    "comment",
    "content",
    "query",
    "question",
    "customer_message"
]


INTENT_KEYWORDS = [
    "intent",
    "intent_name",
    "issue_type",
    "issue",
    "topic",
    "label"
]


RESPONSE_KEYWORDS = [
    "response",
    "reply",
    "answer",
    "resolution",
    "agent_response"
]


BRAND_KEYWORDS = [
    "brand",
    "company",
    "seller",
    "manufacturer"
]


message_col = detect_column(
    df.columns,
    MESSAGE_KEYWORDS
)

intent_col = detect_column(
    df.columns,
    INTENT_KEYWORDS
)

response_col = detect_column(
    df.columns,
    RESPONSE_KEYWORDS
)

brand_col = detect_column(
    df.columns,
    BRAND_KEYWORDS
)


print("\nDetected columns:")

print(
    "Message:",
    message_col
)

print(
    "Intent:",
    intent_col
)

print(
    "Response:",
    response_col
)

print(
    "Brand:",
    brand_col
)


# ================================================================
# FALLBACK MESSAGE COLUMN
# ================================================================

if message_col is None:

    text_candidates = []

    for col in df.columns:

        if df[col].dtype == "object":

            values = (
                df[col]
                .dropna()
                .astype(str)
            )

            if len(values) > 0:

                avg_length = (
                    values.str.len()
                    .mean()
                )

                text_candidates.append(
                    (
                        col,
                        avg_length
                    )
                )


    if not text_candidates:

        raise ValueError(
            "No text column found."
        )


    text_candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )


    message_col = (
        text_candidates[0][0]
    )


    print(
        "\nAutomatically selected:",
        message_col
    )


# ================================================================
# CLEAN DATA
# ================================================================

df["message"] = (
    df[message_col]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True
    )
    .str.strip()
)


# Remove empty messages

df = df[
    df["message"].str.len() >= 3
].copy()


# Remove invalid text

invalid = [
    "",
    "nan",
    "none",
    "null",
    "n/a",
    "na",
    "unknown"
]


df = df[
    ~df["message"]
    .str.lower()
    .isin(invalid)
].copy()


# Remove duplicate messages

df = df.drop_duplicates(
    subset=["message"]
)


df = df.reset_index(
    drop=True
)


print(
    "\nCleaned dataset:",
    df.shape
)


# ================================================================
# STEP 3 — SELECT BRAND
# ================================================================

print("\n")
print("=" * 70)
print("STEP 3 — SELECT ONE BRAND")
print("=" * 70)


if brand_col is not None:

    brand_counts = (
        df[brand_col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .value_counts()
    )


    print(
        "\nTop brands:"
    )

    print(
        brand_counts.head(20)
    )


    SELECTED_BRAND = (
        brand_counts.index[0]
    )


    df_brand = df[
        df[brand_col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        ==
        SELECTED_BRAND
    ].copy()


else:

    SELECTED_BRAND = "All Data"

    df_brand = df.copy()


df_brand = (
    df_brand
    .reset_index(
        drop=True
    )
)


print(
    "\nSelected brand:",
    SELECTED_BRAND
)

print(
    "Records:",
    len(df_brand)
)


# ================================================================
# STEP 4 — DEFINE INTENTS
# ================================================================

print("\n")
print("=" * 70)
print("STEP 4 — DEFINE INTENTS")
print("=" * 70)


DEFAULT_INTENTS = [

    "refund",

    "payment_problem",

    "account_problem",

    "login_problem",

    "delivery_problem",

    "cancellation",

    "subscription",

    "technical_issue",

    "product_problem",

    "complaint",

    "information_request",

    "other"

]


if intent_col is not None:

    values = (
        df_brand[intent_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    values = [
        x for x in values
        if x
        and x.lower()
        not in [
            "nan",
            "none",
            "null",
            "unknown"
        ]
    ]


    if len(set(values)) >= 2:

        INTENTS = list(
            dict.fromkeys(
                values
            )
        )[:12]

    else:

        INTENTS = DEFAULT_INTENTS

else:

    INTENTS = DEFAULT_INTENTS


print(
    "\nIntent categories:"
)

for i, intent in enumerate(
    INTENTS,
    1
):

    print(
        f"{i}. {intent}"
    )


# ================================================================
# STEP 5 — BUILD INTENT MODEL
# ================================================================

print("\n")
print("=" * 70)
print("STEP 5 — BUILD INTENT MODEL")
print("=" * 70)


MODEL_AVAILABLE = False


if intent_col is not None:

    model_df = df_brand.copy()


    model_df["intent"] = (
        model_df[intent_col]
        .fillna("")
        .astype(str)
        .str.strip()
    )


    model_df = model_df[
        model_df["intent"].str.len() > 0
    ]


    counts = (
        model_df["intent"]
        .value_counts()
    )


    valid_classes = (
        counts[
            counts >= 2
        ].index
    )


    model_df = model_df[
        model_df["intent"]
        .isin(valid_classes)
    ]


    if (
        len(model_df) >= 20
        and
        model_df["intent"].nunique() >= 2
    ):

        X = model_df["message"]

        y = model_df["intent"]


        X_train, X_test, y_train, y_test = (
            train_test_split(
                X,
                y,
                test_size=0.20,
                random_state=42,
                stratify=y
            )
        )


        # --------------------------------------------------------
        # BASELINE 1 — MAJORITY CLASS
        # --------------------------------------------------------

        majority = (
            y_train
            .value_counts()
            .index[0]
        )


        majority_pred = [
            majority
            for _ in y_test
        ]


        majority_accuracy = (
            accuracy_score(
                y_test,
                majority_pred
            )
        )


        majority_f1 = (
            f1_score(
                y_test,
                majority_pred,
                average="macro",
                zero_division=0
            )
        )


        print(
            "\nBASELINE 1 — MAJORITY CLASS"
        )

        print(
            "Accuracy:",
            round(
                majority_accuracy,
                4
            )
        )

        print(
            "Macro F1:",
            round(
                majority_f1,
                4
            )
        )


        # --------------------------------------------------------
        # BASELINE 2 — TF-IDF + LOGISTIC REGRESSION
        # --------------------------------------------------------

        intent_vectorizer = TfidfVectorizer(

            lowercase=True,

            strip_accents="unicode",

            ngram_range=(1, 2),

            min_df=1,

            token_pattern=r"(?u)\b\w+\b"
        )


        X_train_vec = (
            intent_vectorizer
            .fit_transform(
                X_train
            )
        )


        X_test_vec = (
            intent_vectorizer
            .transform(
                X_test
            )
        )


        intent_model = (
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced"
            )
        )


        intent_model.fit(
            X_train_vec,
            y_train
        )


        predictions = (
            intent_model
            .predict(
                X_test_vec
            )
        )


        accuracy = (
            accuracy_score(
                y_test,
                predictions
            )
        )


        precision = (
            precision_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            )
        )


        recall = (
            recall_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            )
        )


        macro_f1 = (
            f1_score(
                y_test,
                predictions,
                average="macro",
                zero_division=0
            )
        )


        print(
            "\nBASELINE 2 — TF-IDF + LOGISTIC REGRESSION"
        )

        print(
            "Accuracy:",
            round(
                accuracy,
                4
            )
        )

        print(
            "Precision:",
            round(
                precision,
                4
            )
        )

        print(
            "Recall:",
            round(
                recall,
                4
            )
        )

        print(
            "Macro F1:",
            round(
                macro_f1,
                4
            )
        )


        print(
            "\nClassification Report:"
        )

        print(
            classification_report(
                y_test,
                predictions,
                zero_division=0
            )
        )


        joblib.dump(
            intent_model,
            "intent_model.pkl"
        )


        joblib.dump(
            intent_vectorizer,
            "intent_vectorizer.pkl"
        )


        MODEL_AVAILABLE = True


    else:

        print(
            "Not enough labelled data."
        )

else:

    print(
        "No intent column detected."
    )

    print(
        "Intent model will use keyword rules."
    )


# ================================================================
# STEP 6 — HISTORICAL RETRIEVAL
# ================================================================

print("\n")
print("=" * 70)
print("STEP 6 — HISTORICAL RETRIEVAL")
print("=" * 70)


# ALL usable records are used.

kb_df = df_brand.copy()


kb_messages = (
    kb_df["message"]
    .fillna("")
    .astype(str)
    .tolist()
)


# IMPORTANT FIX:
#
# token_pattern allows one-character tokens.
#
# min_df=1 ensures small datasets work.
#

retrieval_vectorizer = TfidfVectorizer(

    lowercase=True,

    strip_accents="unicode",

    ngram_range=(1, 2),

    min_df=1,

    token_pattern=r"(?u)\b\w+\b"
)


try:

    kb_matrix = (
        retrieval_vectorizer
        .fit_transform(
            kb_messages
        )
    )

except ValueError:

    print(
        "Word TF-IDF failed."
    )

    print(
        "Using character TF-IDF fallback."
    )


    retrieval_vectorizer = TfidfVectorizer(

        analyzer="char",

        ngram_range=(2, 5),

        min_df=1,

        max_features=100000
    )


    kb_matrix = (
        retrieval_vectorizer
        .fit_transform(
            kb_messages
        )
    )


print(
    "Knowledge base:",
    len(kb_messages)
)

print(
    "Matrix:",
    kb_matrix.shape
)


joblib.dump(
    retrieval_vectorizer,
    "retrieval_vectorizer.pkl"
)


# ================================================================
# RETRIEVAL FUNCTION
# ================================================================

def retrieve_cases(
    query,
    top_k=5
):

    query = str(
        query
    ).strip()


    if not query:

        return []


    query_vector = (
        retrieval_vectorizer
        .transform(
            [query]
        )
    )


    if query_vector.nnz == 0:

        return []


    similarities = (
        cosine_similarity(
            query_vector,
            kb_matrix
        )
        .flatten()
    )


    top_indices = (
        np.argsort(
            similarities
        )[::-1][:top_k]
    )


    results = []


    for index in top_indices:

        row = kb_df.iloc[
            index
        ]


        result = {

            "message":
                str(
                    row["message"]
                ),

            "similarity":
                float(
                    similarities[index]
                )
        }


        if response_col is not None:

            result[
                "response"
            ] = str(
                row[
                    response_col
                ]
            )


        if intent_col is not None:

            result[
                "intent"
            ] = str(
                row[
                    intent_col
                ]
            )


        results.append(
            result
        )


    return results


# ================================================================
# STEP 7 — LOCAL REPLY GENERATOR
# ================================================================

print("\n")
print("=" * 70)
print("STEP 7 — GENERATE REPLY")
print("=" * 70)


# ---------------------------------------------------------------
# INTENT PREDICTION
# ---------------------------------------------------------------

def predict_intent(
    message
):

    # ------------------------------------------------------------
    # ML MODEL
    # ------------------------------------------------------------

    if MODEL_AVAILABLE:

        vector = (
            intent_vectorizer
            .transform(
                [message]
            )
        )


        probabilities = (
            intent_model
            .predict_proba(
                vector
            )[0]
        )


        best_index = np.argmax(
            probabilities
        )


        intent = (
            intent_model
            .classes_[best_index]
        )


        confidence = float(
            probabilities[
                best_index
            ]
        )


        return (
            intent,
            confidence
        )


    # ------------------------------------------------------------
    # KEYWORD INTENT CLASSIFIER
    # ------------------------------------------------------------

    text = (
        message
        .lower()
    )


    keyword_rules = {

        "refund": [
            "refund",
            "money back",
            "reimburse",
            "return money"
        ],

        "payment_problem": [
            "payment",
            "paid",
            "charged",
            "charge",
            "transaction",
            "card"
        ],

        "account_problem": [
            "account",
            "profile",
            "account problem"
        ],

        "login_problem": [
            "login",
            "log in",
            "sign in",
            "password"
        ],

        "delivery_problem": [
            "delivery",
            "deliver",
            "shipping",
            "shipment",
            "arrive",
            "package",
            "order"
        ],

        "cancellation": [
            "cancel",
            "cancellation"
        ],

        "subscription": [
            "subscription",
            "subscribe",
            "membership"
        ],

        "technical_issue": [
            "bug",
            "error",
            "not working",
            "broken",
            "technical"
        ],

        "product_problem": [
            "wrong product",
            "damaged",
            "defective",
            "product problem",
            "missing item"
        ],

        "complaint": [
            "complaint",
            "bad service",
            "terrible",
            "unhappy",
            "angry"
        ],

        "information_request": [
            "how",
            "what",
            "where",
            "when",
            "information"
        ]

    }


    scores = {}


    for intent, keywords in (
        keyword_rules.items()
    ):

        score = sum(
            1
            for keyword in keywords
            if keyword in text
        )

        scores[intent] = score


    best_intent = max(
        scores,
        key=scores.get
    )


    best_score = (
        scores[best_intent]
    )


    if best_score == 0:

        return (
            "other",
            0.30
        )


    confidence = min(
        0.55
        +
        0.10 * best_score,
        0.90
    )


    return (
        best_intent,
        confidence
    )


# ================================================================
# REPLY TEMPLATES
# ================================================================

REPLY_TEMPLATES = {

    "refund":
        "We understand that you are requesting a refund. "
        "We will review your request and the relevant order "
        "details. If additional information is required, "
        "our support team will contact you.",


    "payment_problem":
        "We understand that you are having an issue with "
        "a payment or charge. Please provide the relevant "
        "transaction or order details so the support team "
        "can investigate.",


    "account_problem":
        "We understand that you are experiencing an account "
        "issue. Please provide the relevant account details "
        "through the appropriate support channel so we can "
        "investigate.",


    "login_problem":
        "We understand that you are having trouble logging "
        "in. Please check your login details and use the "
        "available password recovery option. If the issue "
        "continues, support can investigate further.",


    "delivery_problem":
        "We understand that you are experiencing a delivery "
        "or shipping issue. Please provide your order details "
        "so the delivery status can be checked.",


    "cancellation":
        "We understand that you would like to cancel your "
        "request or order. Please provide the relevant order "
        "details so the cancellation can be reviewed.",


    "subscription":
        "We understand that you have a question about your "
        "subscription. Please provide the relevant account "
        "details so the subscription can be reviewed.",


    "technical_issue":
        "We understand that you are experiencing a technical "
        "issue. Please provide the relevant details and any "
        "error message so the issue can be investigated.",


    "product_problem":
        "We understand that there is a problem with the "
        "product. Please provide your order details and "
        "information about the issue so it can be reviewed.",


    "complaint":
        "We are sorry to hear about your experience. "
        "Your concern can be reviewed by our support team. "
        "Please provide the relevant details so we can "
        "investigate.",


    "information_request":
        "We understand your question. Please provide any "
        "additional details needed to identify the relevant "
        "product, order, or account so we can assist you.",


    "other":
        "Thank you for contacting support. We need some "
        "additional information to understand your request. "
        "Your message can be reviewed by our support team."

}


# ================================================================
# GROUNDED RESPONSE GENERATOR
# ================================================================

def generate_reply(
    message,
    intent,
    evidence
):

    # ------------------------------------------------------------
    # If historical response exists and evidence is strong,
    # use the historical response as grounding.
    # ------------------------------------------------------------

    if evidence:

        best = evidence[0]


        if (
            "response" in best
            and
            best["response"].strip()
            and
            best["similarity"] >= 0.30
        ):

            historical_response = (
                best["response"]
                .strip()
            )


            return (
                historical_response,
                True
            )


    # ------------------------------------------------------------
    # Otherwise use safe local template.
    # ------------------------------------------------------------

    reply = REPLY_TEMPLATES.get(
        intent,
        REPLY_TEMPLATES["other"]
    )


    return (
        reply,
        False
    )


# ================================================================
# STEP 8 — AUTO-HANDLE / ESCALATE
# ================================================================

print("\n")
print("=" * 70)
print("STEP 8 — AUTO-HANDLE / ESCALATE")
print("=" * 70)


def support_agent(
    message
):

    intent, confidence = (
        predict_intent(
            message
        )
    )


    evidence = (
        retrieve_cases(
            message,
            top_k=5
        )
    )


    best_similarity = (
        evidence[0]["similarity"]
        if evidence
        else 0
    )


    reply, historical_grounding = (
        generate_reply(
            message,
            intent,
            evidence
        )
    )


    # ------------------------------------------------------------
    # DECISION RULES
    # ------------------------------------------------------------

    decision = "AUTO_HANDLE"

    reasons = []


    if confidence < 0.60:

        decision = "ESCALATE"

        reasons.append(
            "Low intent confidence"
        )


    if not evidence:

        decision = "ESCALATE"

        reasons.append(
            "No historical evidence"
        )


    if best_similarity < 0.10:

        decision = "ESCALATE"

        reasons.append(
            "Weak historical similarity"
        )


    if intent == "other":

        decision = "ESCALATE"

        reasons.append(
            "Unknown intent"
        )


    if not historical_grounding:

        # Template responses are safe but not historical
        # evidence-grounded, so escalation is preferred.

        decision = "ESCALATE"

        reasons.append(
            "No strong historical response evidence"
        )


    if not reasons:

        reasons.append(
            "High confidence and strong historical evidence"
        )


    return {

        "message":
            message,

        "intent":
            intent,

        "intent_confidence":
            round(
                confidence,
                4
            ),

        "reply":
            reply,

        "decision":
            decision,

        "grounded":
            historical_grounding,

        "retrieval_similarity":
            round(
                best_similarity,
                4
            ),

        "reason":
            "; ".join(
                reasons
            ),

        "evidence":
            evidence

    }


# ================================================================
# DEMO
# ================================================================

demo_message = (
    "I received the wrong product "
    "and I want a refund."
)


demo = support_agent(
    demo_message
)


print(
    "\nCUSTOMER:"
)

print(
    demo["message"]
)


print(
    "\nINTENT:"
)

print(
    demo["intent"]
)


print(
    "\nCONFIDENCE:"
)

print(
    demo["intent_confidence"]
)


print(
    "\nREPLY:"
)

print(
    demo["reply"]
)


print(
    "\nDECISION:"
)

print(
    demo["decision"]
)


print(
    "\nGROUNDED:"
)

print(
    demo["grounded"]
)


print(
    "\nREASON:"
)

print(
    demo["reason"]
)


# ================================================================
# RUN AGENT ON SAMPLE DATA
# ================================================================

print("\n")
print("=" * 70)
print("RUNNING SUPPORT AGENT")
print("=" * 70)


TEST_SIZE = min(
    20,
    len(df_brand)
)


test_sample = (
    df_brand
    .sample(
        TEST_SIZE,
        random_state=42
    )
)


agent_results = []


for _, row in tqdm(
    test_sample.iterrows(),
    total=len(test_sample)
):

    result = support_agent(
        row["message"]
    )


    agent_results.append({

        "message":
            result["message"],

        "predicted_intent":
            result["intent"],

        "confidence":
            result[
                "intent_confidence"
            ],

        "reply":
            result["reply"],

        "decision":
            result["decision"],

        "grounded":
            result["grounded"],

        "retrieval_similarity":
            result[
                "retrieval_similarity"
            ],

        "reason":
            result["reason"]

    })


results_df = pd.DataFrame(
    agent_results
)


display(
    results_df
)


results_df.to_csv(
    "agent_results.csv",
    index=False
)


# ================================================================
# STEP 9 — CREATE GOLDEN SET
# ================================================================

print("\n")
print("=" * 70)
print("STEP 9 — CREATE GOLDEN SET")
print("=" * 70)


GOLDEN_SIZE = min(
    200,
    len(df_brand)
)


golden_set = (
    df_brand
    .sample(
        GOLDEN_SIZE,
        random_state=123
    )
    [["message"]]
    .copy()
)


# Human labels

golden_set[
    "true_intent"
] = ""


golden_set[
    "expected_handling"
] = ""


golden_set[
    "expected_reply"
] = ""


# Reference label only

if intent_col is not None:

    golden_set[
        "dataset_intent_reference"
    ] = (
        df_brand.loc[
            golden_set.index,
            intent_col
        ]
        .fillna("")
        .astype(str)
        .values
    )


golden_set.to_csv(
    "golden_set_200.csv",
    index=False
)


print(
    "\nGolden set:",
    GOLDEN_SIZE
)


print(
    "\nIMPORTANT:"
)

print(
    "Manually fill true_intent,"
)

print(
    "expected_handling,"
)

print(
    "and expected_reply."
)


display(
    golden_set.head(10)
)


# ================================================================
# STEP 10 — EVALUATION
# ================================================================

print("\n")
print("=" * 70)
print("STEP 10 — EVALUATION")
print("=" * 70)


evaluation = []


if MODEL_AVAILABLE:

    evaluation.append({

        "Model":
            "Majority Class",

        "Accuracy":
            majority_accuracy,

        "Precision":
            np.nan,

        "Recall":
            np.nan,

        "Macro F1":
            majority_f1

    })


    evaluation.append({

        "Model":
            "TF-IDF + Logistic Regression",

        "Accuracy":
            accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "Macro F1":
            macro_f1

    })


evaluation_df = pd.DataFrame(
    evaluation
)


if len(evaluation_df) > 0:

    display(
        evaluation_df
    )

else:

    print(
        "No supervised evaluation available."
    )


print(
    "\nAgent average confidence:",
    round(
        results_df[
            "confidence"
        ].mean(),
        4
    )
)


print(
    "\nAverage retrieval similarity:",
    round(
        results_df[
            "retrieval_similarity"
        ].mean(),
        4
    )
)


print(
    "\nDecision distribution:"
)

print(
    results_df[
        "decision"
    ].value_counts()
)


# ================================================================
# STEP 11 — FAILURE ANALYSIS
# ================================================================

print("\n")
print("=" * 70)
print("STEP 11 — FAILURE ANALYSIS")
print("=" * 70)


failure_analysis = pd.DataFrame({

    "Failure Mode": [

        "Low intent confidence",

        "Weak historical retrieval",

        "No historical response",

        "Ambiguous message",

        "Unknown intent"

    ],

    "Problem": [

        "Classifier is uncertain.",

        "Historical messages are not sufficiently similar.",

        "The dataset does not contain a usable historical response.",

        "One customer message may contain multiple issues.",

        "Message does not fit the predefined intent categories."

    ],

    "Why It Fails": [

        "Intent classes may overlap.",

        "TF-IDF mainly matches lexical similarity.",

        "The dataset may not contain resolution information.",

        "Single-intent classification is limited.",

        "Intent taxonomy may be incomplete."

    ],

    "Improvement": [

        "Add more labelled examples.",

        "Use semantic embeddings or hybrid retrieval.",

        "Add more resolved support conversations.",

        "Support multi-intent classification.",

        "Expand and refine intent categories."

    ]

})


display(
    failure_analysis
)


failure_analysis.to_csv(
    "failure_analysis.csv",
    index=False
)


# ================================================================
# STEP 12 — README
# ================================================================

print("\n")
print("=" * 70)
print("STEP 12 — README")
print("=" * 70)


if MODEL_AVAILABLE:

    metrics_text = f"""
Majority Class Accuracy:
{majority_accuracy:.4f}

Majority Class Macro F1:
{majority_f1:.4f}

TF-IDF + Logistic Regression Accuracy:
{accuracy:.4f}

TF-IDF + Logistic Regression Precision:
{precision:.4f}

TF-IDF + Logistic Regression Recall:
{recall:.4f}

TF-IDF + Logistic Regression Macro F1:
{macro_f1:.4f}
"""

else:

    metrics_text = """
Supervised intent metrics are not available because
the uploaded dataset does not contain sufficient
ground-truth intent labels.
"""


README = f"""
# Hiver SDE Intern Assignment
# AI Customer Support Agent

## Project

This project implements a local AI customer-support
agent without using the OpenAI API.

## Important

NO API KEY IS REQUIRED.

NO OPENAI API IS USED.

The system runs locally using:

- Python
- Pandas
- Scikit-learn
- TF-IDF
- Logistic Regression
- Cosine Similarity
- Rule-based safe response generation

---

# Pipeline

Customer Message
        |
        v
Data Cleaning
        |
        v
Intent Classification
        |
        v
Historical Retrieval
        |
        v
Relevant Past Cases
        |
        v
Reply Generation
        |
        v
Confidence + Evidence
        |
        v
AUTO-HANDLE / ESCALATE

---

# Dataset

Dataset:
{CSV_FILE}

Selected Brand:
{SELECTED_BRAND}

Usable Records:
{len(df_brand)}

---

# Intent Categories

{chr(10).join("- " + x for x in INTENTS)}

---

# Intent Model

The primary local classifier is:

TF-IDF + Logistic Regression

Baselines include:

1. Majority Class
2. TF-IDF + Logistic Regression

---

# Historical Retrieval

The system uses all usable records as a historical
knowledge base.

TF-IDF is used to represent customer messages.

Cosine similarity is used to retrieve similar cases.

---

# Reply Generation

Responses are generated locally.

If a historical response exists and similarity is
strong enough, the historical response can be reused.

Otherwise, a safe intent-based response template is used.

The system does not invent:

- Refund amounts
- Delivery dates
- Discounts
- Company policies
- Compensation
- Completed actions

---

# Auto Handle / Escalate

The system escalates when:

- Confidence < 0.60
- Historical evidence is missing
- Similarity is weak
- Intent is unknown
- No strong historical response is available

---

# Golden Set

A 200-example golden-set template is created.

Human labelling is required for:

- true_intent
- expected_handling
- expected_reply

The golden set should be manually reviewed before
final evaluation.

---

# Evaluation

{metrics_text}

Additional evaluation:

- Retrieval similarity
- Confidence
- Auto-handle rate
- Escalation rate
- Grounding
- Failure analysis

---

# Failure Analysis

The main failure modes considered are:

1. Low intent confidence
2. Weak retrieval
3. Missing historical response
4. Ambiguous messages
5. Unknown intent

---

# Misleading Headline Number

Classification accuracy alone is not sufficient to
measure an AI support agent.

A classifier can correctly identify an intent while
still producing an unsuitable support response.

Therefore this project considers:

- Intent accuracy
- Retrieval quality
- Evidence
- Reply grounding
- Escalation
- Failure modes

---

# Future Improvements

- Sentence embeddings
- Hybrid retrieval
- Semantic search
- Better intent taxonomy
- Multi-intent classification
- More human-labelled data
- LLM-as-judge
- Human agreement
- Confidence calibration

---

# API KEY

No OpenAI API key is required.

No secret key is stored in this project.

---

# Run

Install dependencies:

pip install -r requirements.txt

Then run the notebook/script.

"""


with open(
    "README.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        README
    )


# ================================================================
# DECISION LOG
# ================================================================

decision_log = """
# Hiver SDE Intern — Decision Log

1. No OpenAI API is used.

2. The solution runs locally.

3. All usable records are used as the retrieval knowledge base.

4. Empty messages are removed.

5. Duplicate messages are removed.

6. TF-IDF + Logistic Regression is used as the main
   supervised baseline when intent labels are available.

7. Majority class is used as a simple baseline.

8. Cosine similarity is used for historical retrieval.

9. Multiple historical examples are retrieved.

10. Historical responses are reused only when sufficient
    evidence exists.

11. Safe response templates are used when historical
    responses are unavailable.

12. Low-confidence predictions are escalated.

13. Weak retrieval is escalated.

14. Unknown intents are escalated.

15. A 200-example golden-set template is created.

16. Golden-set labels must be manually verified.

17. Intent accuracy is not treated as the only quality metric.

18. Failure analysis is included.

19. API keys and secrets are excluded from the repository.
"""


with open(
    "decision_log.md",
    "w",
    encoding="utf-8"
) as f:

    f.write(
        decision_log
    )


# ================================================================
# STEP 13 — CREATE GITHUB PROJECT
# ================================================================

print("\n")
print("=" * 70)
print("STEP 13 — CREATE GITHUB REPOSITORY")
print("=" * 70)


PROJECT = (
    "hiver-support-agent"
)


if os.path.exists(
    PROJECT
):

    shutil.rmtree(
        PROJECT
    )


folders = [

    PROJECT,

    f"{PROJECT}/data",

    f"{PROJECT}/data/raw",

    f"{PROJECT}/data/processed",

    f"{PROJECT}/data/golden",

    f"{PROJECT}/notebooks",

    f"{PROJECT}/src",

    f"{PROJECT}/evaluation",

    f"{PROJECT}/models",

    f"{PROJECT}/tests"

]


for folder in folders:

    os.makedirs(
        folder,
        exist_ok=True
    )


# ================================================================
# COPY FILES
# ================================================================

shutil.copy(
    CSV_FILE,
    f"{PROJECT}/data/raw/{CSV_FILE}"
)


df_brand.to_csv(
    f"{PROJECT}/data/processed/cleaned_data.csv",
    index=False
)


shutil.copy(
    "golden_set_200.csv",
    f"{PROJECT}/data/golden/golden_set_200.csv"
)


shutil.copy(
    "failure_analysis.csv",
    f"{PROJECT}/evaluation/failure_analysis.csv"
)


shutil.copy(
    "agent_results.csv",
    f"{PROJECT}/evaluation/agent_results.csv"
)


shutil.copy(
    "README.md",
    f"{PROJECT}/README.md"
)


shutil.copy(
    "decision_log.md",
    f"{PROJECT}/decision_log.md"
)


# ================================================================
# COPY MODELS
# ================================================================

for model_file in [

    "intent_model.pkl",

    "intent_vectorizer.pkl",

    "retrieval_vectorizer.pkl"

]:

    if os.path.exists(
        model_file
    ):

        shutil.copy(
            model_file,
            f"{PROJECT}/models/{model_file}"
        )


# ================================================================
# REQUIREMENTS
# ================================================================

requirements = """
pandas
numpy
scikit-learn
tqdm
joblib
"""


with open(
    f"{PROJECT}/requirements.txt",
    "w"
) as f:

    f.write(
        requirements
    )


# ================================================================
# GITIGNORE
# ================================================================

gitignore = """
__pycache__/
*.pyc
.ipynb_checkpoints/

.env
.env.*
*.key
secrets.json

# Do not commit secrets

*.pkl

# Exclude large model files if necessary
"""


with open(
    f"{PROJECT}/.gitignore",
    "w"
) as f:

    f.write(
        gitignore
    )


# ================================================================
# CONFIG
# ================================================================

config = {

    "dataset":
        CSV_FILE,

    "selected_brand":
        str(
            SELECTED_BRAND
        ),

    "records":
        int(
            len(df_brand)
        ),

    "intents":
        INTENTS,

    "intent_model":
        "TF-IDF + Logistic Regression",

    "retrieval":
        "TF-IDF + cosine similarity",

    "reply_generation":
        "Local template + historical response",

    "llm":
        False,

    "openai_api":
        False,

    "api_key_required":
        False,

    "confidence_threshold":
        0.60,

    "golden_set":
        GOLDEN_SIZE

}


with open(
    f"{PROJECT}/config.json",
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=4
    )


# ================================================================
# PROJECT SUMMARY
# ================================================================

summary = f"""
HIVER SDE INTERN ASSIGNMENT
============================

Project:
AI Customer Support Agent

Dataset:
{CSV_FILE}

Selected Brand:
{SELECTED_BRAND}

Records:
{len(df_brand)}

Intents:
{len(INTENTS)}

Intent Model:
TF-IDF + Logistic Regression

Retrieval:
TF-IDF + Cosine Similarity

Reply Generation:
Local template + historical response

OpenAI API:
NOT USED

API Key:
NOT REQUIRED

Confidence Threshold:
0.60

Golden Set:
{GOLDEN_SIZE}

Pipeline:

1. Get Dataset
2. Analyze Data
3. Select Brand
4. Define Intents
5. Build Intent Model
6. Historical Retrieval
7. Generate Reply
8. Auto-handle / Escalate
9. Create Golden Set
10. Evaluate
11. Failure Analysis
12. Report + README
13. GitHub Repository
"""


with open(
    f"{PROJECT}/PROJECT_SUMMARY.txt",
    "w"
) as f:

    f.write(
        summary
    )


# ================================================================
# ZIP PROJECT
# ================================================================

ZIP_PATH = shutil.make_archive(
    PROJECT,
    "zip",
    PROJECT
)


# ================================================================
# FINAL RESULT
# ================================================================

print("\n")
print("=" * 70)
print("COMPLETE PROJECT CREATED")
print("=" * 70)


print(
    "\nProject:",
    PROJECT
)


print(
    "\nZIP:",
    ZIP_PATH
)


print(
    "\nOpenAI API:",
    "NOT USED"
)


print(
    "API KEY:",
    "NOT REQUIRED"
)


print("\nProject structure:")


for root, dirs, file_list in os.walk(
    PROJECT
):

    level = root.replace(
        PROJECT,
        ""
    ).count(
        os.sep
    )


    indent = (
        "    "
        * level
    )


    print(
        indent
        +
        os.path.basename(
            root
        )
        +
        "/"
    )


    for filename in file_list:

        print(
            indent
            +
            "    "
            +
            filename
        )


print("\n")
print("=" * 70)
print("DOWNLOAD")
print("=" * 70)


files.download(
    ZIP_PATH
)

HIVER SDE INTERN ASSIGNMENT
LOCAL AI CUSTOMER SUPPORT AGENT
NO OPENAI API


STEP 1 — GET DATASET
Upload your CSV dataset.


Saving amazon_product.csv to amazon_product (2).csv

Selected dataset:
amazon_product (2).csv
Loaded using: utf-8

Dataset shape: (64, 22)


STEP 2 — ANALYZE DATA

Rows: 64
Columns: 22

Column names:
- Unnamed: 0
- asin
- product_title
- product_price
- product_original_price
- currency
- product_star_rating
- product_num_ratings
- product_url
- product_photo
- product_num_offers
- product_minimum_offer_price
- is_best_seller
- is_amazon_choice
- is_prime
- climate_pledge_friendly
- sales_volume
- delivery
- has_variations
- product_availability
- unit_price
- unit_count

Missing values:
product_availability           63
unit_price                     59
unit_count                     59
product_original_price         37
product_star_rating            10
sales_volume                    3
delivery                        1
currency                        0
product_title                   0
asin                            0
Unnamed: 0                      0
product_price                  

  0%|          | 0/20 [00:00<?, ?it/s]

,message,predicted_intent,confidence,reply,decision,grounded,retrieval_similarity,reason
0,https://m.media-amazon.com/images/I/71kmhd1Otd...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
1,https://m.media-amazon.com/images/I/61bMNCeAUA...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
2,https://m.media-amazon.com/images/I/71rP7f78eF...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
3,https://m.media-amazon.com/images/I/61SUj2aKoE...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
4,https://m.media-amazon.com/images/I/61s0ZzwzSC...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
5,https://m.media-amazon.com/images/I/81Mya-dPIO...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
6,https://m.media-amazon.com/images/I/41b3zsiq4p...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
7,https://m.media-amazon.com/images/I/71xzMSijzk...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
8,https://m.media-amazon.com/images/I/617GUqQEIC...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...
9,https://m.media-amazon.com/images/I/71OznGGxhc...,other,0.3,Thank you for contacting support. We need some...,ESCALATE,False,1.0,Low intent confidence; Unknown intent; No stro...




STEP 9 — CREATE GOLDEN SET

Golden set: 40

IMPORTANT:
Manually fill true_intent,
expected_handling,
and expected_reply.


...



STEP 10 — EVALUATION
No supervised evaluation available.

Agent average confidence: 0.3

Average retrieval similarity: 1.0

Decision distribution:
decision
ESCALATE    20
Name: count, dtype: int64


STEP 11 — FAILURE ANALYSIS


,Failure Mode,Problem,Why It Fails,Improvement
0,Low intent confidence,Classifier is uncertain.,Intent classes may overlap.,Add more labelled examples.
1,Weak historical retrieval,Historical messages are not sufficiently similar.,TF-IDF mainly matches lexical similarity.,Use semantic embeddings or hybrid retrieval.
2,No historical response,The dataset does not contain a usable historic...,The dataset may not contain resolution informa...,Add more resolved support conversations.
3,Ambiguous message,One customer message may contain multiple issues.,Single-intent classification is limited.,Support multi-intent classification.
4,Unknown intent,Message does not fit the predefined intent cat...,Intent taxonomy may be incomplete.,Expand and refine intent categories.




STEP 12 — README


STEP 13 — CREATE GITHUB REPOSITORY


COMPLETE PROJECT CREATED

Project: hiver-support-agent

ZIP: /content/hiver-support-agent.zip

OpenAI API: NOT USED
API KEY: NOT REQUIRED

Project structure:
hiver-support-agent/
    decision_log.md
    config.json
    README.md
    PROJECT_SUMMARY.txt
    .gitignore
    requirements.txt
    data/
        raw/
            amazon_product (2).csv
        golden/
            golden_set_200.csv
        processed/
            cleaned_data.csv
    models/
        retrieval_vectorizer.pkl
    notebooks/
    tests/
    evaluation/
        agent_results.csv
        failure_analysis.csv
    src/


DOWNLOAD


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>